# Mini Project 2 — Part 3

**NetID: dpate172**

This notebook selects commits around each project's longest gap, summarizes
their themes, and adds the explanations, status, and reflections required in
Parts 3.1 and 3.2. All inputs are saved in the repository, so rerunning it does
not need API access.

## Selections and interpretation

- Gap samples use the last ten commits before the gap and first ten after it,
  ordered by the Part 1 author timestamp in UTC, then SHA for ties.
- NodePy and Geoportal each have only two post-gap records. FHIR Server and
  cuCIM have no empty months, so their gap answers are N/A and they have no
  rows in the gap CSV. All ten projects have statistics and reflections.
- Each message has one reviewed primary theme, saved with a short reason.
  The top two are ranked by count; ties follow the README's category order.
  If only one theme is supported, that single theme is retained. Identifiable
  merges inherit their described change; generic merges are Other. Bot-authored
  updates use Automated bot contributions.
- Explanations distinguish observed changes from possible causes. Returning
  authors are checked against all dated pre-gap records, using raw identities.

**Status date:** the README explicitly uses **September 30, 2025**, with
April 1, 2025 as the activity threshold. Recent themes use the latest ten
commits from the reconstructed default-branch history through that cutoff,
ordered by committer time. This differs from the later Part 1 GitHub snapshot
and from the author dates used for gaps. A present-day Git history cannot
reconstruct every past force-push or exactly when a commit became public.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display
from scripts.analyze_part3 import analyze_part3, read_csv, select_gap_commits

assert Path("dpate172_project_summary.csv").exists(), "Run from the repository folder."
pd.set_option("display.max_colwidth", None)
stats, gap_rows, recent_rows, report = analyze_part3()
stats_df = pd.DataFrame(stats)
print(f"{report['project_count']} projects; {len(gap_rows)} gap-sample commits; {len(recent_rows)} recent commits.")
print("Created the gap CSV, updated statistics, and ten reflection files.")

10 projects; 144 gap-sample commits; 100 recent commits.
Created the gap CSV, updated statistics, and ten reflection files.


## Gap samples

The exported CSV keeps the original five fields and adds `pre/post` first.
The 13 records without timestamps remain in Part 1, but cannot enter a
time-based sample. Their missing dates are documented in the affected projects.

In [2]:
counts = pd.DataFrame([
    {"Project": project, "Before": item["pre_count"], "After": item["post_count"],
     "Recent": item["recent_count"], "Undated source rows": item["omitted_undated"]}
    for project, item in report["projects"].items()
])
display(counts.set_index("Project"))
display(stats_df.set_index("Project")[["BeforeThemes", "AfterThemes"]])

,Before,After,Recent,Undated source rows
Project,,,,
ketch_nodepy,10,2,10,0
mouseland_suite2p,10,10,10,0
esri_geoportal-server,10,2,10,0
hpparvi_pytransit,10,10,10,0
sundmanbo_opencalphad,10,10,10,0
stack-of-tasks_pinocchio,10,10,10,0
microsoft_fhir-server,0,0,10,12
rapidsai_cucim,0,0,10,0
agnwinds_python,10,10,10,1


,BeforeThemes,AfterThemes
Project,,
ketch_nodepy,Documentation updates,Bug fixes:Documentation updates
mouseland_suite2p,Bug fixes:Feature development,Other:Bug fixes
esri_geoportal-server,Automated bot contributions:Documentation updates,Documentation updates
hpparvi_pytransit,Other:Documentation updates,Other:Bug fixes
sundmanbo_opencalphad,Feature development:Bug fixes,Other:Feature development
stack-of-tasks_pinocchio,Bug fixes:Feature development,Other:Documentation updates
microsoft_fhir-server,N/A active Project,N/A active Project
rapidsai_cucim,N/A active Project,N/A active Project
agnwinds_python,Bug fixes:Other,Feature development:Documentation updates


## What happened around the gaps

The linked reflections give the evidence behind each interpretation. A return
to committing can be brief or limited to documentation; it does not necessarily
show that full development resumed.

In [3]:
for row in stats:
    project = row["Project"]
    display(Markdown(f"**[{project}](dpate172_reflection_{project}.md)** — {row['Notes']}"))

**[ketch_nodepy](dpate172_reflection_ketch_nodepy.md)** — NodePy has an 11-month gap after a run of documentation maintenance. David Ketcheson later corrected numerical coefficients, but only two post-gap commits are available. This supports a small return to maintenance, while the exact reason for the pause remains unclear.

**[mouseland_suite2p](dpate172_reflection_mouseland_suite2p.md)** — Suite2p has one empty month, followed by fixes to previously reported bugs. Chris Ki drives the immediate return, with the same raw identities already present earlier in the project. The issue discussions explain the work that resumed, but not why committing paused during November.

**[esri_geoportal-server](dpate172_reflection_esri_geoportal-server.md)** — Geoportal Server v1 is explicitly retired, which makes its eight-month gap easier to explain. The two later changes update the README and website instead of restarting application development. Its July 2025 README commit still qualifies it as Active under the assignment's six-month rule, showing the limit of that rule.

**[hpparvi_pytransit](dpate172_reflection_hpparvi_pytransit.md)** — The seven-month gap is difficult to explain because the pre-gap messages do not announce a stop. Activity resumes with fixes for the z == k bug in issue 12, followed by other numerical maintenance. The same author supplies all ten sampled recovery commits, and the dataset contains 967 dated commits after the gap.

**[sundmanbo_opencalphad](dpate172_reflection_sundmanbo_opencalphad.md)** — The five-month gap is consistent with the maintainer’s documented part-time development during retirement, although that does not prove the exact cause. Bo Sundman resumes model implementation in June 2024, and a new raw author identity later supplies build and CI work. Recovery is intermittent: the first ten post-gap commits extend into October.

**[stack-of-tasks_pinocchio](dpate172_reflection_stack-of-tasks_pinocchio.md)** — The longest zero run is only December 2014, so Pinocchio has no inactivity gap of at least three months. Returning contributors resume teaching-lab updates and routine development, but no inspected source explains the brief pause. The first selected commit has a January author date and March committer date, so the author-time plot is not an integration timeline.

**[microsoft_fhir-server](dpate172_reflection_microsoft_fhir-server.md)** — FHIR Server has commits in all 88 observed months, so there is no gap to explain or recovery to attribute. Activity is broadly steady across the main part of the timeline. The partial final month and 12 missing timestamps limit comparisons at the endpoint.

**[rapidsai_cucim](dpate172_reflection_rapidsai_cucim.md)** — cuCIM has commits in all 63 observed months even though its yearly totals decline overall. Activity rises again in 2025, so lower volume does not mean the project became inactive. No gap or recovery cohort can be identified from this monthly series.

**[agnwinds_python](dpate172_reflection_agnwinds_python.md)** — The two-month zero run is below the three-month inactivity threshold, and its cause is undocumented. Returning contributors resume radiation-hydrodynamics features, documentation, and fixes; preserved PR 1002 confirms the feature work. One missing timestamp could change the measured gap, and the later Sirocco rename should not be mistaken for its cause.

**[csgillespie_powerlaw](dpate172_reflection_csgillespie_powerlaw.md)** — The 25-month gap has no documented cause in the inspected sources, and recovery looks like occasional package and documentation maintenance. A vignette fix is author-dated January 2024, but its PR is merged in February 2025, illustrating the difference between author time and integration time. The apparent new author string belongs to an automated site deployment using the existing maintainer’s email.

## Status and recent themes

Active means at least one eligible default-branch commit during April–September
2025. Inactive means the most recent eligible commit predates that interval.
This is a date rule, so a retired project can still be Active after a README edit.
Full recent messages, dates, theme labels, and reasons are saved in
`data/dpate172_recent_commits.csv`.

In [4]:
status = stats_df[["Project", "Currentstatus", "RecentThemes"]].copy()
status["Last commit as of cutoff (UTC)"] = status["Project"].map(
    lambda project: report["projects"][project]["latest_github_commit_as_of_cutoff"])
display(status.set_index("Project"))

,Currentstatus,RecentThemes,Last commit as of cutoff (UTC)
Project,,,
ketch_nodepy,Inactive,Documentation updates:Bug fixes,2024-09-04T07:27:22Z
mouseland_suite2p,Active,Bug fixes:Feature development,2025-08-30T18:56:25Z
esri_geoportal-server,Active,Documentation updates:Dependency updates,2025-07-08T20:25:48Z
hpparvi_pytransit,Active,Bug fixes:Documentation updates,2025-07-22T11:45:24Z
sundmanbo_opencalphad,Active,Other:Bug fixes,2025-05-13T09:11:01Z
stack-of-tasks_pinocchio,Active,Dependency updates:Automated bot contributions,2025-09-29T07:08:59Z
microsoft_fhir-server,Active,Bug fixes:Dependency updates,2025-09-30T15:35:55Z
rapidsai_cucim,Active,Automated bot contributions:Dependency updates,2025-09-24T15:40:04Z
agnwinds_python,Inactive,Bug fixes:Documentation updates,2024-10-22T21:47:15Z


## Checks

Confirm that the gap CSV preserves the selected source records, all ten projects
have reflections, and the first fourteen statistics columns are unchanged.
The independent validator also checks boundary cases and selection order.

In [5]:
from scripts.validate_part3 import validate_part3
validate_part3(Path.cwd())
print("Part 3 checks passed.")

Logic checks passed: UTC boundaries, ties, sample sizes, no gaps, themes, and author history.
Final checks passed: 144 gap rows, 100 recent rows, 10 reflections.
Part 3 checks passed.
